# Item 15d — o veredito do ADR-0006

**A regra vem antes do número.** Esta célula reproduz o pré-compromisso do
[ADR-0006](../docs/adr/0006-precompromisso-fase-atencao.md), aceito em 2026-10-13 antes de
qualquer treino com atenção. Nenhuma célula acima desta lê resultado, e as de baixo só contam e
escolhem: a interpretação já estava escrita.

**Hipóteses**, em cada split, nas 6 células (2 charts × k ∈ {1, 2, 4}), on-chart:

- **H1:** `A` (atenção sobre o grafo real) supera o sem_grafo;
- **H2:** `A` supera o `A-emb` (atenção sobre o grafo embaralhado);
- **H3:** `A` supera a proposta SAGE (informativa, **não decide**).

**Família do Holm:** as 18 comparações (3 contrastes × 6 células) de cada split. **"Vence"** = IC
bootstrap 95% da diferença de erro quadrático por música inteiro abaixo de zero **e** `p` Holm
< 0,05. **"Perde"** é o espelho. **"Maioria"** = 4 de 6. Os testes rodam sobre a média das seeds,
como no item 10, e cada seed isolada é reportada como sensibilidade.

| Desfecho (por split) | Condição | Afirmação |
|---|---|---|
| **O1** | H1 e H2 em maioria | O nulo era do encoder. Com atenção a topologia paga, e a conclusão da dissertação reverte para o caso 1 (sob este encoder), com o SAGE como limitação |
| **O2** | H1 em maioria, H2 não | O ganho vem de agregar atributos dos vizinhos, não da topologia (caso 2 do ADR-0001) |
| **O3** | H1 não está em maioria, `A` não perde em maioria (inclui H2 sem H1) | O nulo é robusto à arquitetura. A conclusão do ADR-0001 sai fortalecida, e isso é resultado, não fracasso. Se H2 valer sem H1, a nota é que o embaralhado prejudica e o grafo real fica neutro em relação a não ter grafo |
| **O4** | `A` perde para o sem_grafo em maioria | A atenção prejudica (sobreajuste da vizinhança). Reportado como está, com o orçamento de ajuste como limitação |

**Precedência:** O4 vale antes dos demais. Se os dois splits discordarem, a discordância é o
resultado (como no ADR-0005), e a afirmação principal fica restrita ao que se replica.

**Limitação declarada:** a proposta SAGE teve 24 configurações de ajuste; a atenção teve 8.

Os limiares, a família, os contrastes e a regra não são digitados neste notebook: vêm de
`music_diffusion_gnn.evaluation.veredito_atencao`, testado em `tests/test_veredito_atencao.py`.

## Entradas e saídas

- `comparacao_final/predicoes_cinco_modelos.parquet` (item 10): sem_grafo e proposta SAGE, já na
  média das seeds;
- `avaliacao_matriz/mode2_{sem_grafo,proposta}_{regime}_seed{seed}_completo.parquet` (item 09): as
  seeds isoladas da referência, para a sensibilidade;
- `atencao/avaliacao/mode2_{atencao,atencao_emb}_*_completo.parquet` (item 15c);
- `atencao/treinos/matriz_val_mse.parquet` (item 15b): a contagem de parâmetros do `A`.

Só a leitura de orçamento `completo` entra, como no item 10. Saída em `atencao/` (no Colab, a
pasta do Drive que corresponde a `results/atencao/`): `veredito.parquet`, `relatorio.md`,
`testes_pareados_atencao.parquet` e as duas tabelas de sensibilidade.

## 0. Ambiente — Colab (GPU) ou local

No Colab, clona o repositório (código + artefatos de dados versionados). Este notebook é só
pandas: **não precisa de GPU** nem de PyG.
Repo privado: cole um PAT em `GITHUB_TOKEN`.

> Os grafos `hetero_full_current.pt` e `hetero_full_pre_pandemia.pt` precisam estar **commitados**
> antes de clonar: os CSVs brutos da rede de gêneros não são versionados, então o grafo não pode
> ser reconstruído aqui dentro.

In [ ]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

DATA_FILES = [
    "data/processed/graph/hetero_full_current.pt",
    "data/processed/graph/hetero_full_pre_pandemia.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if IS_COLAB:
    if Path(REPO_DIR, "pyproject.toml").exists():
        # O runtime sobrevive ao restart do kernel: um clone de sessão anterior ficaria
        # para trás em silêncio e o import viria do código velho. Sincroniza sempre.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    else:
        tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
        print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
        if _clone(url).returncode != 0:
            from getpass import getpass
            tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
            _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
    os.chdir(REPO_DIR)
    print("commit em uso:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
    faltando = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
    print("✓ Colab pronto. cwd =", os.getcwd())
    if faltando:
        print("⚠️ FALTAM no repositório:", faltando)
else:
    print("Local — nada a clonar.")

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

_anchor = Path(globals().get("__vsc_ipynb_file__", os.getcwd()))
if not _anchor.exists():
    _anchor = Path(os.getcwd())
ROOT = _anchor if _anchor.is_dir() else _anchor.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"raiz do projeto não encontrada a partir de {_anchor}"
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from music_diffusion_gnn.evaluation.veredito_atencao import (
    ALPHA, CONTRASTS, FAMILY_SIZE, MAJORITY, N_CELLS, OUTCOMES, TREATED,
    budget_block, classify_outcome, compare_models, sensitivity_by_seed,
)
print("ROOT =", ROOT)

In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/music-influence-gnn")
else:
    BASE = ROOT / "results"
COMPARACAO = BASE / "comparacao_final"
MATRIZ     = BASE / "avaliacao_matriz"
ATENCAO    = BASE / "atencao"
OUT        = ATENCAO
OUT.mkdir(parents=True, exist_ok=True)
print("saída →", OUT)

## 1. A regra, como o módulo a carrega

A célula abaixo só imprime as constantes do módulo, para que o relatório mostre de onde vieram.

In [ ]:
MODELOS = [TREATED, *CONTRASTS.values()]
print(f"tratado: {TREATED}")
for h, c in CONTRASTS.items():
    print(f"  {h}: {TREATED} contra {c}" + ("  (informativa, não decide)" if h == "H3" else ""))
print(f"células por split: {N_CELLS} | família do Holm: {FAMILY_SIZE} | maioria: {MAJORITY} de "
      f"{N_CELLS} | alfa: {ALPHA}")
for code, txt in OUTCOMES.items():
    print(f"  {code}: {txt}")

## 2. As predições dos sete modelos

As três seeds de cada modelo de atenção são reduzidas a uma predição por origem pela **média
entre seeds**, exatamente como o item 10 fez com a escada SAGE. As cinco referências vêm da tabela
do item 10; o teste só usa as que entram nos contrastes.

In [ ]:
CHAVE_ORIGEM = ["song_id", "chart", "origin_week", "k", "split_regime"]

p_cinco = COMPARACAO / "predicoes_cinco_modelos.parquet"
assert p_cinco.exists(), f"{p_cinco} não existe: rode item10_comparacao_consolidada.ipynb"
cinco = pd.read_parquet(p_cinco)

arquivos_att = sorted((ATENCAO / "avaliacao").glob("mode2_*_completo.parquet"))
assert arquivos_att, "nenhum mode2_*_completo.parquet da atenção: rode o item 15c"
att = pd.concat([pd.read_parquet(f) for f in arquivos_att], ignore_index=True)
assert set(att.variante) <= {"atencao", "atencao_emb"}, set(att.variante)

att_medio = (att.groupby(CHAVE_ORIGEM + ["variante"], observed=True)
                .agg(y_true=("y_true", "first"), y_pred=("y_pred", "mean"),
                     onchart=("onchart", "first"), n_seeds=("seed", "nunique"))
                .reset_index()
                .rename(columns={"variante": "modelo"}))
print(att_medio.groupby(["modelo", "split_regime"]).agg(origens=("y_pred", "size"),
                                                       seeds=("n_seeds", "min")))

sete = pd.concat([cinco[CHAVE_ORIGEM + ["modelo", "y_true", "y_pred", "onchart"]],
                  att_medio[CHAVE_ORIGEM + ["modelo", "y_true", "y_pred", "onchart"]]],
                 ignore_index=True)
print("\nmodelos:", sorted(sete.modelo.unique()))
faltam = [m for m in MODELOS if m not in set(sete.modelo)]
assert not faltam, f"modelos ausentes: {faltam}"

## 3. Os 18 testes de cada split

`compare_models` pareia, em cada célula on-chart, o erro quadrático de `A` e da referência nas
mesmas origens, promedia por música, e aplica Wilcoxon e IC bootstrap 95% (`paired_by_song`, a
estatística do item 10). O Holm corre sobre as 18 comparações de cada split. `mean_diff < 0`
significa erro **menor** do `A`.

In [ ]:
testes = compare_models(sete)
assert (testes.groupby("split_regime").size() == FAMILY_SIZE).all()
testes.to_parquet(OUT / "testes_pareados_atencao.parquet", index=False)
display(testes[["split_regime", "hipotese", "contraste", "chart", "k", "mean_diff", "ci_lo",
                "ci_hi", "p_value", "p_holm", "win_rate", "n_songs", "rotulo"]].round(6))

## 4. O desfecho, por split

`classify_outcome` conta e escolhe, com a precedência do ADR (O4 primeiro). A discordância entre
splits, quando existe, é o resultado.

In [ ]:
desfecho = classify_outcome(testes)
display(desfecho)

if desfecho.discordancia.any():
    print("\n⚠️ DISCORDÂNCIA ENTRE SPLITS: " + " | ".join(
        f"{r.split_regime} → {r.desfecho}" for r in desfecho.itertuples()))
    print(f"   {desfecho.nota.iloc[0]}.")
else:
    print(f"\nos dois splits concordam: {desfecho.desfecho.iloc[0]} — {desfecho.afirmacao.iloc[0]}")

# Uma linha por (split, contraste), com o desfecho do split: cada rótulo é recomputável a partir
# de testes_pareados_atencao.parquet.
cont = (testes.groupby(["split_regime", "hipotese", "contraste"])["rotulo"]
              .value_counts().unstack(fill_value=0).reset_index())
for col in ("vence", "perde", "inconclusivo"):
    if col not in cont:
        cont[col] = 0
veredito = cont.rename(columns={"vence": "vitorias", "perde": "derrotas",
                                "inconclusivo": "inconclusivas"}).merge(
    desfecho[["split_regime", "desfecho", "afirmacao", "discordancia", "afirmacao_principal"]],
    on="split_regime")
veredito.to_parquet(OUT / "veredito.parquet", index=False)
display(veredito)

## 5. Sensibilidade por seed isolada

O teste principal roda sobre a média das seeds. Aqui H1 a H3 se repetem em **cada seed isolada**
(seed `s` de `A` e `A-emb` contra a seed `s` do sem_grafo e da proposta SAGE, do item 09), e cada
célula diz quantas seeds concordam com o rótulo da média. O desfecho de cada seed aparece ao lado
do desfecho da média.

In [ ]:
partes = [att[att.orcamento_arestas == "completo"].rename(columns={"variante": "modelo"})]
for modelo in [m for m in MODELOS if m not in {"atencao", "atencao_emb"}]:
    fs = sorted(MATRIZ.glob(f"mode2_{modelo}_*_completo.parquet"))
    assert fs, f"nenhum mode2_{modelo}_*_completo.parquet em {MATRIZ} (item 09)"
    partes.append(pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True)
                    .rename(columns={"variante": "modelo"}))
cols = CHAVE_ORIGEM + ["modelo", "seed", "y_true", "y_pred", "onchart"]
por_seed = pd.concat([p[cols] for p in partes], ignore_index=True)
print(por_seed.groupby(["modelo", "split_regime"]).seed.unique())

sens_celulas, sens_desfecho = sensitivity_by_seed(por_seed, testes)
sens_celulas.to_parquet(OUT / "sensibilidade_seeds_celulas.parquet", index=False)
sens_desfecho.to_parquet(OUT / "sensibilidade_seeds_desfecho.parquet", index=False)
display(sens_desfecho)
display(sens_celulas)

## 6. Parâmetros e orçamento de ajuste

O número de parâmetros do `A` vem dos checkpoints do item 15b; o da proposta SAGE e o orçamento de
ajuste vêm do módulo. Os dois entram no mesmo bloco da tabela de resultados (AC6).

In [ ]:
p_matriz = ATENCAO / "treinos" / "matriz_val_mse.parquet"
assert p_matriz.exists(), f"{p_matriz} não existe: rode a seção 6 do item 15b"
matriz = pd.read_parquet(p_matriz)
n_params_a = matriz.loc[matriz.variante == "atencao", "n_params"].dropna().astype(int).unique()
assert len(n_params_a) == 1, f"o A tem contagens de parâmetros diferentes entre treinos: {n_params_a}"
BLOCO = budget_block(int(n_params_a[0]))
print(BLOCO)

## 7. O relatório

`relatorio.md` junta a regra, a tabela de desfecho por split (com a discordância destacada), a
sensibilidade por seed e o bloco de orçamento, nesta ordem.

In [ ]:
def tabela_md(df: pd.DataFrame) -> str:
    cab = "| " + " | ".join(map(str, df.columns)) + " |"
    sep = "|" + "---|" * len(df.columns)
    def fmt(v):
        return f"{v:.6g}" if isinstance(v, (float, np.floating)) else str(v)
    return "\n".join([cab, sep] + ["| " + " | ".join(fmt(v) for v in r) + " |"
                                   for r in df.itertuples(index=False)])

partes_md = [
    "# Veredito da fase de atenção (ADR-0006)", "",
    "## Regra (pré-compromisso)", "",
    *[f"- **{c}**: {t}" for c, t in OUTCOMES.items()],
    f"- Maioria = {MAJORITY} de {N_CELLS} células; Holm sobre as {FAMILY_SIZE} comparações de cada "
    f"split; vence = IC 95% inteiro abaixo de zero e p Holm < {ALPHA}; O4 tem precedência; "
    "H3 é informativa.", "",
    "## Desfecho por split", "",
    tabela_md(desfecho[["split_regime", "vitorias_h1", "derrotas_h1", "vitorias_h2",
                        "vitorias_h3", "desfecho", "afirmacao"]]), "",
]
if desfecho.discordancia.any():
    partes_md += [f"> **Discordância entre splits:** {desfecho.nota.iloc[0]}.", ""]
else:
    partes_md += [f"Os dois splits concordam: **{desfecho.desfecho.iloc[0]}**.", ""]
partes_md += [
    "## Testes por célula (on-chart, média das seeds)", "",
    tabela_md(testes[["split_regime", "hipotese", "contraste", "chart", "k", "mean_diff",
                      "ci_lo", "ci_hi", "p_holm", "n_songs", "rotulo"]]), "",
    "## Sensibilidade por seed isolada", "",
    tabela_md(sens_desfecho), "",
    tabela_md(sens_celulas), "",
    "## Parâmetros e orçamento de ajuste (limitação)", "",
    BLOCO, "",
]
(OUT / "relatorio.md").write_text("\n".join(partes_md), encoding="utf-8")
print(f"✓ {OUT / 'relatorio.md'} e {OUT / 'veredito.parquet'} gravados")

## 8. O que levar de volta

Do Drive (`atencao/`, o equivalente de `results/atencao/`):

- `veredito.parquet` — uma linha por (split, contraste), com vitórias, derrotas e o desfecho do split;
- `relatorio.md` — o relatório com a regra, o desfecho, a sensibilidade e o orçamento;
- `testes_pareados_atencao.parquet` — os 36 testes, de onde cada rótulo é recomputável;
- `sensibilidade_seeds_{celulas,desfecho}.parquet` — a concordância seed a seed.

Depois disso: reescrever a conclusão segundo a linha da tabela do ADR-0006 que saiu, e só então
decidir se o P3 (ablação no `A`, ATT-07) abre, o que só acontece com O1 ou O2.